# Funding Rate Arbitrage Scanner

Compare BTC funding rates across **Hyperliquid** and **Lighter.xyz** using the [0xArchive SDK](https://pypi.org/project/oxarchive/).

This notebook fetches historical funding rate snapshots from both exchanges and creates:
- **Side-by-side rate comparison** showing how funding diverges across venues
- **Spread analysis** with statistical threshold bands
- **Annualized carry** visualization (APR from rate differential)
- **Arbitrage opportunity windows** — contiguous periods where the spread exceeds 2σ
- **Cumulative hypothetical P&L** from a simple spread-harvesting strategy
- **Distribution analysis** of rate differentials

**Requirements:** Free tier API key from [0xarchive.io/dashboard](https://0xarchive.io/dashboard) (BTC, 30-day lookback)

## 1. Setup

In [ ]:
%pip install oxarchive pandas matplotlib seaborn numpy python-dotenv scipy -q

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats
from oxarchive import Client

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# --- Configuration ---
# Load API key from .env file (copy .env.example to .env and add your key)
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

API_KEY = os.environ.get("OXARCHIVE_API_KEY", "your_api_key_here")
if API_KEY == "your_api_key_here":
    raise ValueError("Set OXARCHIVE_API_KEY in .env or as an environment variable")

COIN = "BTC"
LOOKBACK_DAYS = 7  # Increase for more data (max 30 on Free: history covers the most recent rolling 30 days)

# Exchange colors
COLOR_HL = "#3498db"       # blue — Hyperliquid
COLOR_LT = "#e67e22"       # orange — Lighter
COLOR_SPREAD = "#9b59b6"   # purple — Spread / differential
COLOR_LONG = "#2ecc71"     # green — Long cheaper rate
COLOR_SHORT = "#e74c3c"    # red — Short expensive rate

# Spread threshold: opportunities beyond ±2σ
SIGMA_THRESHOLD = 2

client = Client(api_key=API_KEY)
end = datetime.now(timezone.utc)
start = end - timedelta(days=LOOKBACK_DAYS)
print(f"Scanning {COIN} funding rates: {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} UTC")
print(f"Exchanges: Hyperliquid, Lighter.xyz")

## 2. Fetch Funding Rates

Both exchanges expose continuous funding rate snapshots via the same SDK interface:
- **Hyperliquid** — ~1 minute intervals, reports hourly funding rate
- **Lighter** — ~10 second intervals, reports 8-hour funding rate

We paginate through both and resample to a common hourly frequency.

In [ ]:
# --- Fetch Hyperliquid funding rates ---
hl_records = []
result = client.hyperliquid.funding.history(COIN, start=start, end=end, limit=1000)
hl_records.extend(result.data)

while result.next_cursor:
    result = client.hyperliquid.funding.history(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    hl_records.extend(result.data)
    print(f"\rHyperliquid: fetched {len(hl_records):,} snapshots...", end="", flush=True)

print(f"\rHyperliquid: {len(hl_records):,} funding snapshots over {LOOKBACK_DAYS} days")

In [ ]:
# --- Fetch Lighter funding rates ---
lt_records = []
result = client.lighter.funding.history(COIN, start=start, end=end, limit=1000)
lt_records.extend(result.data)

while result.next_cursor:
    result = client.lighter.funding.history(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    lt_records.extend(result.data)
    print(f"\rLighter: fetched {len(lt_records):,} snapshots...", end="", flush=True)

print(f"\rLighter:     {len(lt_records):,} funding snapshots over {LOOKBACK_DAYS} days")

## 3. Data Processing

Build DataFrames, resample both to hourly means, then normalize to 8-hour funding rates.

**Rate convention:** Hyperliquid reports an hourly rate (×8 for 8h settlement). Lighter reports an 8-hour rate directly. We auto-detect via the ratio of raw means.

In [ ]:
# Build raw DataFrames
hl_df = pd.DataFrame([
    {"timestamp": r.timestamp, "rate": float(r.funding_rate)}
    for r in hl_records
])
hl_df["timestamp"] = pd.to_datetime(hl_df["timestamp"], utc=True)
hl_df = hl_df.set_index("timestamp").sort_index()

lt_df = pd.DataFrame([
    {"timestamp": r.timestamp, "rate": float(r.funding_rate)}
    for r in lt_records
])
lt_df["timestamp"] = pd.to_datetime(lt_df["timestamp"], utc=True)
lt_df = lt_df.set_index("timestamp").sort_index()

# Resample both to hourly means
hl_hourly = hl_df["rate"].resample("1h").mean().dropna()
lt_hourly = lt_df["rate"].resample("1h").mean().dropna()

# Auto-detect rate scaling
hl_mean = hl_hourly.mean()
lt_mean = lt_hourly.mean()
ratio = abs(lt_mean / hl_mean) if abs(hl_mean) > 1e-12 else 1.0

print(f"Raw hourly means — HL: {hl_mean:.10f}, Lighter: {lt_mean:.10f}")
print(f"Lighter / HL ratio: {ratio:.1f}x")

# Normalize to 8-hour rates
# HL reports hourly rate → multiply by 8
# Lighter: if ratio > 30, it's already an 8h rate; otherwise multiply by 8
hl_8h = hl_hourly * 8
if ratio > 30:
    lt_8h = lt_hourly  # already 8h rate
    print("Lighter rate detected as 8-hour rate (used as-is)")
else:
    lt_8h = lt_hourly * 8
    print("Lighter rate detected as hourly rate (×8 for 8h)")

# Align on common hourly index
aligned = pd.DataFrame({"hl_8h": hl_8h, "lt_8h": lt_8h}).dropna()

expected_hours = LOOKBACK_DAYS * 24
coverage = len(aligned) / expected_hours if expected_hours > 0 else 0
if len(aligned) < 10:
    raise ValueError(
        f"Only {len(aligned)} aligned hours — not enough overlapping data. "
        "Check that both exchanges have data in the lookback window."
    )
if coverage < 0.5:
    print(f"⚠ Low coverage: {len(aligned)}/{expected_hours} hours ({coverage:.0%}). "
          "Lighter may not have data for the full window.")

aligned["spread"] = aligned["lt_8h"] - aligned["hl_8h"]

print(f"\nAligned hours: {len(aligned):,} / {expected_hours} ({coverage:.0%} coverage)")
print(f"8h rate range — HL: [{aligned['hl_8h'].min():.6f}, {aligned['hl_8h'].max():.6f}]")
print(f"8h rate range — LT: [{aligned['lt_8h'].min():.6f}, {aligned['lt_8h'].max():.6f}]")
print(f"Spread range:       [{aligned['spread'].min():.6f}, {aligned['spread'].max():.6f}]")

## 4. Funding Rate Comparison

Side-by-side view of 8-hour funding rates on both exchanges. When the lines diverge, there's an arbitrage opportunity.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))

ax.plot(aligned.index, aligned["hl_8h"] * 100, color=COLOR_HL,
        linewidth=1.2, alpha=0.9, label="Hyperliquid")
ax.plot(aligned.index, aligned["lt_8h"] * 100, color=COLOR_LT,
        linewidth=1.2, alpha=0.9, label="Lighter")

ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("8-Hour Funding Rate (%)")
ax.set_title(f"{COIN} Funding Rates — Hyperliquid vs Lighter ({LOOKBACK_DAYS}d)", fontsize=16)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.4f}%"))
ax.legend(loc="upper left", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Rate Differential (Spread)

The spread = Lighter rate − Hyperliquid rate. Dashed lines mark the ±2σ threshold — periods beyond these bands represent statistically significant divergences.

In [ ]:
spread_mean = aligned["spread"].mean()
spread_std = aligned["spread"].std()
upper_thresh = spread_mean + SIGMA_THRESHOLD * spread_std
lower_thresh = spread_mean - SIGMA_THRESHOLD * spread_std

fig, ax = plt.subplots(figsize=(18, 7))

ax.plot(aligned.index, aligned["spread"] * 100, color=COLOR_SPREAD,
        linewidth=1.0, alpha=0.9, label="Spread (LT − HL)")
ax.axhline(spread_mean * 100, color="white", linewidth=0.8, alpha=0.5,
           linestyle="-", label=f"Mean ({spread_mean*100:.5f}%)")
ax.axhline(upper_thresh * 100, color=COLOR_SHORT, linewidth=1.0,
           linestyle="--", alpha=0.8, label=f"+{SIGMA_THRESHOLD}σ ({upper_thresh*100:.5f}%)")
ax.axhline(lower_thresh * 100, color=COLOR_LONG, linewidth=1.0,
           linestyle="--", alpha=0.8, label=f"−{SIGMA_THRESHOLD}σ ({lower_thresh*100:.5f}%)")

# Shade regions beyond thresholds
spread_pct = aligned["spread"] * 100
ax.fill_between(aligned.index, upper_thresh * 100, spread_pct,
                where=spread_pct > upper_thresh * 100,
                color=COLOR_SHORT, alpha=0.2, interpolate=True)
ax.fill_between(aligned.index, lower_thresh * 100, spread_pct,
                where=spread_pct < lower_thresh * 100,
                color=COLOR_LONG, alpha=0.2, interpolate=True)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Spread (%)")
ax.set_title(f"{COIN} Funding Rate Spread — Lighter vs Hyperliquid", fontsize=16)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.4f}%"))
ax.legend(loc="upper left", fontsize=10)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

above = (aligned["spread"] > upper_thresh).sum()
below = (aligned["spread"] < lower_thresh).sum()
total = len(aligned)
print(f"Time above +{SIGMA_THRESHOLD}σ: {above}/{total} hours ({above/total*100:.1f}%)")
print(f"Time below −{SIGMA_THRESHOLD}σ: {below}/{total} hours ({below/total*100:.1f}%)")
print(f"Time within bands:  {total-above-below}/{total} hours ({(total-above-below)/total*100:.1f}%)")

## 6. Annualized Carry

Convert 8-hour rates to annual percentage rates: `APR = 8h_rate × 3 × 365`.

The top panel shows each exchange's annualized rate; the bottom panel shows the annualized spread.

In [ ]:
aligned["hl_apr"] = aligned["hl_8h"] * 3 * 365
aligned["lt_apr"] = aligned["lt_8h"] * 3 * 365
aligned["spread_apr"] = aligned["spread"] * 3 * 365

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

# Top: annualized rates
axes[0].plot(aligned.index, aligned["hl_apr"] * 100, color=COLOR_HL,
             linewidth=1.2, alpha=0.9, label="Hyperliquid APR")
axes[0].plot(aligned.index, aligned["lt_apr"] * 100, color=COLOR_LT,
             linewidth=1.2, alpha=0.9, label="Lighter APR")
axes[0].axhline(0, color="white", linewidth=0.5, alpha=0.3)
axes[0].set_ylabel("Annualized Rate (%)")
axes[0].set_title(f"{COIN} Annualized Funding Rates — {LOOKBACK_DAYS}d Window", fontsize=16)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
axes[0].legend(loc="upper left", fontsize=12)

# Bottom: annualized spread (filled area)
axes[1].fill_between(aligned.index, 0, aligned["spread_apr"] * 100,
                     where=aligned["spread_apr"] >= 0,
                     color=COLOR_LT, alpha=0.4, interpolate=True, label="LT > HL")
axes[1].fill_between(aligned.index, 0, aligned["spread_apr"] * 100,
                     where=aligned["spread_apr"] < 0,
                     color=COLOR_HL, alpha=0.4, interpolate=True, label="HL > LT")
axes[1].axhline(0, color="white", linewidth=0.5, alpha=0.3)
axes[1].set_ylabel("Spread APR (%)")
axes[1].set_xlabel("Time (UTC)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
axes[1].legend(loc="upper left", fontsize=10)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 7. Arbitrage Opportunity Windows

Detect contiguous periods where the absolute spread exceeds the 2σ threshold. Each window represents a potential arbitrage opportunity — go long on the exchange with the lower rate, short on the higher.

In [ ]:
# Detect opportunity windows: |spread| > threshold
aligned["is_opportunity"] = (
    (aligned["spread"] > upper_thresh) | (aligned["spread"] < lower_thresh)
)

# Group contiguous True blocks into windows
aligned["window_id"] = (
    aligned["is_opportunity"] != aligned["is_opportunity"].shift()
).cumsum()

windows = []
for wid, grp in aligned[aligned["is_opportunity"]].groupby("window_id"):
    duration = (grp.index[-1] - grp.index[0]).total_seconds() / 3600
    direction = "LT > HL" if grp["spread"].mean() > 0 else "HL > LT"
    windows.append({
        "start": grp.index[0],
        "end": grp.index[-1],
        "duration_hours": max(duration, 1),  # single-bar windows count as 1h
        "avg_spread": grp["spread"].mean(),
        "max_spread": grp["spread"].abs().max(),
        "direction": direction,
    })

windows_df = pd.DataFrame(windows)
print(f"Detected {len(windows_df)} arbitrage windows\n")

# Visualize windows on spread chart
fig, ax = plt.subplots(figsize=(18, 7))

ax.plot(aligned.index, aligned["spread"] * 100, color=COLOR_SPREAD,
        linewidth=0.8, alpha=0.8)
ax.axhline(upper_thresh * 100, color=COLOR_SHORT, linewidth=1.0,
           linestyle="--", alpha=0.6)
ax.axhline(lower_thresh * 100, color=COLOR_LONG, linewidth=1.0,
           linestyle="--", alpha=0.6)

for _, w in windows_df.iterrows():
    color = COLOR_SHORT if w["direction"] == "LT > HL" else COLOR_LONG
    ax.axvspan(w["start"], w["end"], color=color, alpha=0.15)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Spread (%)")
ax.set_title(f"{COIN} Arbitrage Opportunity Windows (±{SIGMA_THRESHOLD}σ)", fontsize=16)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.4f}%"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Top 10 windows table
if len(windows_df) > 0:
    top_windows = windows_df.sort_values("duration_hours", ascending=False).head(10)
    print("Top 10 Opportunity Windows (by duration):")
    print("-" * 90)
    for _, w in top_windows.iterrows():
        print(
            f"  {w['start']:%Y-%m-%d %H:%M} -> {w['end']:%Y-%m-%d %H:%M} UTC"
            f" | {w['duration_hours']:>3.0f}h"
            f" | avg spread {w['avg_spread']*100:>+.5f}%"
            f" | max |spread| {w['max_spread']*100:.5f}%"
            f" | {w['direction']}"
        )
else:
    print("No arbitrage windows detected at this threshold.")

## 8. Cumulative Hypothetical P&L

**Strategy:** During each opportunity window hour, collect `|spread| / 8` (the per-hour share of the 8h settlement). This simulates being long on the cheaper-rate exchange and short on the more-expensive one.

*This is illustrative only — ignores fees, slippage, and execution risk.*

In [ ]:
# Per-hour P&L: collect |spread|/8 during opportunity windows
aligned["hourly_pnl"] = 0.0
mask = aligned["is_opportunity"]
aligned.loc[mask, "hourly_pnl"] = aligned.loc[mask, "spread"].abs() / 8
aligned["cum_pnl"] = aligned["hourly_pnl"].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2]})

# Top: per-hour P&L bars (in basis points)
pnl_bps = aligned["hourly_pnl"] * 10_000
bar_colors = np.where(aligned["spread"] > 0, COLOR_LT, COLOR_HL)
axes[0].bar(aligned.index, pnl_bps, width=pd.Timedelta("1h"),
            color=bar_colors, alpha=0.7)
axes[0].set_ylabel("Hourly P&L (bps)")
axes[0].set_title(f"{COIN} Hypothetical Arbitrage P&L", fontsize=16)

# Bottom: cumulative P&L line (%)
axes[1].fill_between(aligned.index, 0, aligned["cum_pnl"] * 100,
                     color=COLOR_SPREAD, alpha=0.3)
axes[1].plot(aligned.index, aligned["cum_pnl"] * 100, color=COLOR_SPREAD,
             linewidth=2.0)
axes[1].set_ylabel("Cumulative P&L (%)")
axes[1].set_xlabel("Time (UTC)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

active_hours = mask.sum()
total_pnl = aligned["cum_pnl"].iloc[-1]
annualized_pnl = total_pnl * (365 / LOOKBACK_DAYS) if LOOKBACK_DAYS > 0 else 0
print(f"Active hours (in opportunity windows): {active_hours}/{len(aligned)} ({active_hours/len(aligned)*100:.1f}%)")
print(f"Total hypothetical P&L:    {total_pnl*100:.4f}%  ({total_pnl*10_000:.2f} bps)")
print(f"Annualized hypothetical:   {annualized_pnl*100:.2f}%")

## 9. Distribution Analysis

How are funding rate spreads distributed? The histogram reveals whether opportunities cluster around zero or exhibit fat tails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: 8h spread histogram
spread_pct_vals = aligned["spread"] * 100
axes[0].hist(spread_pct_vals, bins=50, color=COLOR_SPREAD, alpha=0.7, edgecolor="none")
axes[0].axvline(spread_mean * 100, color="white", linewidth=1.5, linestyle="--", alpha=0.8)
axes[0].axvline(upper_thresh * 100, color=COLOR_SHORT, linewidth=1.0, linestyle="--", alpha=0.6)
axes[0].axvline(lower_thresh * 100, color=COLOR_LONG, linewidth=1.0, linestyle="--", alpha=0.6)
axes[0].set_xlabel("8h Spread (%)")
axes[0].set_ylabel("Count (hours)")
axes[0].set_title("8-Hour Spread Distribution", fontsize=14)

# Stats box
skew = sp_stats.skew(aligned["spread"].values)
kurt = sp_stats.kurtosis(aligned["spread"].values)
stats_text = (
    f"Mean: {spread_mean*100:.5f}%\n"
    f"Std:  {spread_std*100:.5f}%\n"
    f"Skew: {skew:.3f}\n"
    f"Kurt: {kurt:.3f}"
)
axes[0].text(0.97, 0.97, stats_text, transform=axes[0].transAxes,
             verticalalignment="top", horizontalalignment="right",
             fontsize=10, fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.5))

# Right: annualized spread histogram
apr_pct_vals = aligned["spread_apr"] * 100
axes[1].hist(apr_pct_vals, bins=50, color=COLOR_SPREAD, alpha=0.7, edgecolor="none")
apr_mean = aligned["spread_apr"].mean() * 100
apr_std = aligned["spread_apr"].std() * 100
axes[1].axvline(apr_mean, color="white", linewidth=1.5, linestyle="--", alpha=0.8)
axes[1].set_xlabel("Annualized Spread (%)")
axes[1].set_ylabel("Count (hours)")
axes[1].set_title("Annualized Spread Distribution", fontsize=14)

apr_stats_text = (
    f"Mean: {apr_mean:.2f}%\n"
    f"Std:  {apr_std:.2f}%\n"
    f"Skew: {skew:.3f}\n"
    f"Kurt: {kurt:.3f}"
)
axes[1].text(0.97, 0.97, apr_stats_text, transform=axes[1].transAxes,
             verticalalignment="top", horizontalalignment="right",
             fontsize=10, fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.5))

plt.tight_layout()
plt.show()

## 10. Summary Statistics

In [ ]:
print(f"{'='*70}")
print(f"  {COIN} FUNDING RATE ARBITRAGE SUMMARY — {LOOKBACK_DAYS} Day Window")
print(f"  {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} UTC")
print(f"  Exchanges: Hyperliquid, Lighter.xyz")
print(f"{'='*70}")

print(f"\n  Data Points")
print(f"  {'-'*40}")
print(f"  Hyperliquid snapshots:  {len(hl_records):>12,}")
print(f"  Lighter snapshots:      {len(lt_records):>12,}")
print(f"  Aligned hourly bars:    {len(aligned):>12,}")

print(f"\n  8-Hour Funding Rates")
print(f"  {'-'*40}")
print(f"  HL mean (8h):           {aligned['hl_8h'].mean()*100:>11.5f}%")
print(f"  LT mean (8h):           {aligned['lt_8h'].mean()*100:>11.5f}%")
print(f"  Spread mean:            {spread_mean*100:>+11.5f}%")
print(f"  Spread std:             {spread_std*100:>11.5f}%")

print(f"\n  Annualized Rates")
print(f"  {'-'*40}")
print(f"  HL mean APR:            {aligned['hl_apr'].mean()*100:>11.2f}%")
print(f"  LT mean APR:            {aligned['lt_apr'].mean()*100:>11.2f}%")
print(f"  Spread mean APR:        {aligned['spread_apr'].mean()*100:>+11.2f}%")
print(f"  Spread std APR:         {aligned['spread_apr'].std()*100:>11.2f}%")

print(f"\n  Arbitrage Opportunities (±{SIGMA_THRESHOLD}σ threshold)")
print(f"  {'-'*40}")
print(f"  Windows detected:       {len(windows_df):>12}")
print(f"  Total opportunity hours:{active_hours:>12}")
print(f"  % of time in window:    {active_hours/len(aligned)*100:>11.1f}%")
if len(windows_df) > 0:
    print(f"  Longest window:         {windows_df['duration_hours'].max():>11}h")
    print(f"  Avg window duration:    {windows_df['duration_hours'].mean():>11.1f}h")

print(f"\n  Hypothetical P&L (illustrative only)")
print(f"  {'-'*40}")
print(f"  Total P&L ({LOOKBACK_DAYS}d):        {total_pnl*100:>11.4f}%")
print(f"  Total P&L (bps):        {total_pnl*10_000:>11.2f}")
print(f"  Annualized P&L:         {annualized_pnl*100:>11.2f}%")
print(f"{'='*70}")

In [ ]:
client.close()
print("Client closed. Done!")